# N038 · 归并排序与逆序计数

**目标：** 从有序子问题合并中同时恢复答案和统计量。

**先修：** N037, N013, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 分治；归并；递归树；稳定性；逆序对。

## 从问题到算法

归并排序先递归排序两半，再用两个指针线性合并。若右值先被取走，则尚未取走的左半值都比它大，一次贡献左半剩余长度个逆序对。翻转对条件是 a[i]>2a[j]，不能直接复用普通逆序数量；在合并前单独做单调扫描。

## 最小实现

**本章接口：** `merge_sort(a)`、`count_inversions(a)`、`reverse_pairs(a)`

In [1]:
def merge_sort(a):
    if len(a)<2: return list(a)
    mid=len(a)//2; left=merge_sort(a[:mid]); right=merge_sort(a[mid:])
    out=[]; i=j=0
    while i<len(left) and j<len(right):
        if left[i]<=right[j]: out.append(left[i]); i+=1
        else: out.append(right[j]); j+=1
    return out+left[i:]+right[j:]

def _count_pairs(a,factor):
    if len(a)<2: return list(a),0
    mid=len(a)//2
    left,count1=_count_pairs(a[:mid],factor)
    right,count2=_count_pairs(a[mid:],factor)
    count=count1+count2; j=0
    for x in left:
        while j<len(right) and x>factor*right[j]: j+=1
        count+=j
    out=[]; i=j=0
    while i<len(left) and j<len(right):
        if left[i]<=right[j]: out.append(left[i]); i+=1
        else: out.append(right[j]); j+=1
    return out+left[i:]+right[j:],count

def count_inversions(a): return _count_pairs(a,1)[1]
def reverse_pairs(a): return _count_pairs(a,2)[1]

## 正确性、前提与复杂度

索引对唯一分为左半内部、右半内部、跨两半三类。因factor为正，右半有序使满足x>factor*y的y构成前缀；x递增时此前缀不缩短。

**代价：** 三种主算法 O(n log n) 时间、O(n) 峰值辅助空间及 O(log n) 递归深度。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,3,2,3,1]
show_table(['i','j','a[i]','a[j]','普通逆序','翻转对'],[(i,j,a[i],a[j],a[i]>a[j],a[i]>2*a[j]) for i in range(len(a)) for j in range(i+1,len(a))])

i,j,a[i],a[j],普通逆序,翻转对
0,1,1,3,False,False
0,2,1,2,False,False
0,3,1,3,False,False
0,4,1,1,False,False
1,2,3,2,True,False
1,3,3,3,False,False
1,4,3,1,True,True
2,3,2,3,False,False
2,4,2,1,True,False
3,4,3,1,True,True


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert count_inversions([3,1,2])==2
assert reverse_pairs([1,3,2,3,1])==2
from itertools import product
for n in range(6):
    for a in product([-2,-1,0,1,2],repeat=n):
        assert merge_sort(a)==sorted(a)
        assert count_inversions(a)==sum(a[i]>a[j] for i in range(n) for j in range(i+1,n))
        assert reverse_pairs(a)==sum(a[i]>2*a[j] for i in range(n) for j in range(i+1,n))
print('N038: 所有本章断言通过')

N038: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 区分普通逆序对与a[i]>2*a[j]。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用索引区间减少切片。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 912. Sort an Array（canonical）
- 493. Reverse Pairs（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。